# Setup

In [27]:
!pip install \
  pyarrow==21.0.0 \
  azure-storage-file-datalake==12.21.0 \
  azure-identity==1.24.0

Looking in indexes: https://pypi.org/simple, http://nxd-pip-registry/index/


In [28]:
from datetime import datetime
from io import BytesIO
from uuid import uuid4

from nxd.data_product.client import create_client
import pyarrow as pa
import pyarrow.parquet as pq
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
from nxd.data_product.context import AzureDataLakeStorage

# Helper Functions

In [23]:
def _get_adls_client(context: AzureDataLakeStorage) -> DataLakeServiceClient:
    credentials = ClientSecretCredential(
        context.tenant_id,
        context.client_id,
        context.client_secret,
    )
    return DataLakeServiceClient(
        f"https://{context.account_name}.dfs.core.windows.net", credential=credentials
    )


def parquet_to_adls(
    context: AzureDataLakeStorage,
    table: pa.Table,
    file_path: str,
) -> None:
    client = _get_adls_client(context)

    file_client = client.get_file_client(
        file_system=context.container,
        file_path=file_path,
    )

    buffer = BytesIO()
    pq.write_table(table, buffer)
    buffer.seek(0)

    file_client.upload_data(buffer, overwrite=True)


def adls_to_parquet(
    context: AzureDataLakeStorage,
    file_path: str,
) -> pa.Table:
    client = _get_adls_client(context)

    file_client = client.get_file_client(
        file_system=context.container,
        file_path=file_path,
    )

    file = file_client.download_file().readall()
    reader = pa.BufferReader(file)

    return pq.read_table(reader)

def delete_adls_file(
    context: AzureDataLakeStorage,
    file_path: str,
) -> None:
    client = _get_adls_client(context)

    file_client = client.get_file_client(
        file_system=context.container,
        file_path=file_path,
    )

    file_client.delete_file()


In [33]:
def append_to_credit_card_tx(
    customers_pylist: list,
    transactions_pylist: list,
    context: AzureDataLakeStorage,
) -> None:
    customers = [adls_to_parquet(context, adls.model_paths["customers"].path)]
    transactions = [adls_to_parquet(context, adls.model_paths["transactions"].path)]

    transactions.append(pa.Table.from_pylist(transactions_pylist))
    customers.append(pa.Table.from_pylist(customers_pylist))
    
    transactions = pa.concat_tables(transactions)
    customers = pa.concat_tables(customers)
    
    parquet_to_adls(context, transactions, context.model_paths["transactions"].path)
    parquet_to_adls(context, customers, context.model_paths["customers"].path)

In [ ]:
nxd_client = create_client(hostname="dp.demo.trynxd.com")
data_product = nxd_client.data_product(data_product="credit-card-tx-demo")
adls = data_product.get_outputs("adls", AzureDataLakeStorage)

# Data Injection

## Fail `credit-card-tx-pii-compliance` Promise

Insert an unrefacted email into the `customers` model, causing the `credit-card-tx-pii-compliance` Promise to fail

In [35]:
customer_id = str(uuid4())
customers_pylist = [
    {
        "customer_id": customer_id,
        # unredacted email
        "email": "sneaky_email@personal.com",
    },
]

transactions_pylist = [
    {
        "transaction_id": str(uuid4()),
        "customer_id": customer_id,
        "date": datetime(2025, 3, 5, 12, 55),
        "type": "payment",
        "amount": 4399,
        "merchant": "Serious Snags",
        "country_code": "AU",
    },
    {
        "transaction_id": str(uuid4()),
        "customer_id": customer_id,
        "date": datetime(2025, 6, 11, 8, 3),
        "type": "deposit",
        "amount": 1000,
        "merchant": None,
        "country_code": "AU",
    },
]

append_to_credit_card_tx(
    customers_pylist,
    transactions_pylist,
    context=adls,
)

## Provide suspicious Transactions for downstream Data Product `suspicious-tx`

Insert transaction records for the same customer in multiple countries for use within the downstream Data Product `suspicious-tx`

In [38]:
customer_id = str(uuid4())
customers_pylist = [
    {
        "customer_id": customer_id,
        "email": "personal.com",
    },
]

transactions_pylist = [
    {
        "transaction_id": str(uuid4()),
        "customer_id": customer_id,
        "date": datetime(2025, 7, 4, 9, 10),
        "type": "payment",
        "amount": 1999,
        "merchant": "Bogan Bargins",
        "country_code": "AU",
    },
    {
        "transaction_id": str(uuid4()),
        "customer_id": customer_id,
        "date": datetime(2025, 7, 4, 9, 35),
        "type": "payment",
        "amount": 20000,
        "merchant": "Dubious Dacquoises",
        "country_code": "FR",
    },
    {
        "transaction_id": str(uuid4()),
        "customer_id": customer_id,
        "date": datetime(2025, 7, 4, 9, 50),
        "type": "transfer",
        "amount": 5000,
        "merchant": None,
        "country_code": "AU",
    },
]

append_to_credit_card_tx(
    customers_pylist,
    transactions_pylist,
    context=adls,
)

## Reset Models

Clear both the `customers` and `transactions` models, the next transformation run will restore them with correct, promise passing data.

In [39]:
delete_adls_file(
    context=adls,
    file_path=adls.model_paths["customers"].path,
)

delete_adls_file(
    context=adls,
    file_path=adls.model_paths["transactions"].path,
)